In [621]:
# 客户端初始化
import voyageai
import json
# 加载环境变量并创建客户端
from dotenv import load_dotenv
from anthropic import Anthropic

embedding_client = voyageai.Client()


load_dotenv()
base_url = "http://127.0.0.1:8045"
api_key = "sk-5b2137d3a6c74956a519669d86aa4e71"
client = Anthropic(api_key=api_key, base_url=base_url)
model = "gemini-3-flash"
# model = "claude-opus-4-5-thinking"

In [622]:
# 辅助函数
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    for block in message.content:
        if block.type == "text":
            if stop_sequences:
                for stop_seq in stop_sequences:
                    if stop_seq in block.text:
                        block.text = block.text.split(stop_seq)[0]
                        break   
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [623]:
# 按章节分块
import re


def chunk_by_section(document_text):
    pattern = r"\n## "
    return re.split(pattern, document_text)

In [624]:
# 嵌入向量生成
def generate_embedding(chunks, model="voyage-3-large", input_type="query"):
    is_list = isinstance(chunks, list)
    input = chunks if is_list else [chunks]
    result = embedding_client.embed(input, model=model, input_type=input_type)
    return result.embeddings if is_list else result.embeddings[0]

In [625]:
# VectorIndex 实现
import math
from typing import Optional, Any, List, Dict, Tuple


class VectorIndex:
    def __init__(
        self,
        distance_metric: str = "cosine",
        embedding_fn=None,
    ):
        self.vectors: List[List[float]] = []
        self.documents: List[Dict[str, Any]] = []
        self._vector_dim: Optional[int] = None
        if distance_metric not in ["cosine", "euclidean"]:
            raise ValueError("distance_metric must be 'cosine' or 'euclidean'")
        self._distance_metric = distance_metric
        self._embedding_fn = embedding_fn

    def add_document(self, document: Dict[str, Any]):
        if not self._embedding_fn:
            raise ValueError("Embedding function not provided during initialization.")
        if not isinstance(document, dict):
            raise TypeError("Document must be a dictionary.")
        if "content" not in document:
            raise ValueError("Document dictionary must contain a 'content' key.")

        content = document["content"]
        if not isinstance(content, str):
            raise TypeError("Document 'content' must be a string.")

        vector = self._embedding_fn(content)
        self.add_vector(vector=vector, document=document)

    def add_documents(self, documents: List[Dict[str, Any]]):
        if not self._embedding_fn:
            raise ValueError("Embedding function not provided during initialization.")

        if not isinstance(documents, list):
            raise TypeError("Documents must be a list of dictionaries.")

        if not documents:
            return

        contents = []
        for i, doc in enumerate(documents):
            if not isinstance(doc, dict):
                raise TypeError(f"Document at index {i} must be a dictionary.")
            if "content" not in doc:
                raise ValueError(f"Document at index {i} must contain a 'content' key.")
            if not isinstance(doc["content"], str):
                raise TypeError(f"Document 'content' at index {i} must be a string.")
            contents.append(doc["content"])

        vectors = self._embedding_fn(contents)

        for vector, document in zip(vectors, documents):
            self.add_vector(vector=vector, document=document)

    def search(self, query: Any, k: int = 1) -> List[Tuple[Dict[str, Any], float]]:
        if not self.vectors:
            return []

        if isinstance(query, str):
            if not self._embedding_fn:
                raise ValueError("Embedding function not provided for string query.")
            query_vector = self._embedding_fn(query)
        elif isinstance(query, list) and all(
            isinstance(x, (int, float)) for x in query
        ):
            query_vector = query
        else:
            raise TypeError("Query must be either a string or a list of numbers.")

        if self._vector_dim is None:
            return []

        if len(query_vector) != self._vector_dim:
            raise ValueError(
                f"Query vector dimension mismatch. Expected {self._vector_dim}, got {len(query_vector)}"
            )

        if k <= 0:
            raise ValueError("k must be a positive integer.")

        if self._distance_metric == "cosine":
            dist_func = self._cosine_distance
        else:
            dist_func = self._euclidean_distance

        distances = []
        for i, stored_vector in enumerate(self.vectors):
            distance = dist_func(query_vector, stored_vector)
            distances.append((distance, self.documents[i]))

        distances.sort(key=lambda item: item[0])

        return [(doc, dist) for dist, doc in distances[:k]]

    def add_vector(self, vector, document: Dict[str, Any]):
        if not isinstance(vector, list) or not all(
            isinstance(x, (int, float)) for x in vector
        ):
            raise TypeError("Vector must be a list of numbers.")
        if not isinstance(document, dict):
            raise TypeError("Document must be a dictionary.")
        if "content" not in document:
            raise ValueError("Document dictionary must contain a 'content' key.")

        if not self.vectors:
            self._vector_dim = len(vector)
        elif len(vector) != self._vector_dim:
            raise ValueError(
                f"Inconsistent vector dimension. Expected {self._vector_dim}, got {len(vector)}"
            )

        self.vectors.append(list(vector))
        self.documents.append(document)

    def _euclidean_distance(self, vec1: List[float], vec2: List[float]) -> float:
        if len(vec1) != len(vec2):
            raise ValueError("Vectors must have the same dimension")
        return math.sqrt(sum((p - q) ** 2 for p, q in zip(vec1, vec2)))

    def _dot_product(self, vec1: List[float], vec2: List[float]) -> float:
        if len(vec1) != len(vec2):
            raise ValueError("Vectors must have the same dimension")
        return sum(p * q for p, q in zip(vec1, vec2))

    def _magnitude(self, vec: List[float]) -> float:
        return math.sqrt(sum(x * x for x in vec))

    def _cosine_distance(self, vec1: List[float], vec2: List[float]) -> float:
        if len(vec1) != len(vec2):
            raise ValueError("Vectors must have the same dimension")

        mag1 = self._magnitude(vec1)
        mag2 = self._magnitude(vec2)

        if mag1 == 0 and mag2 == 0:
            return 0.0
        elif mag1 == 0 or mag2 == 0:
            return 1.0

        dot_prod = self._dot_product(vec1, vec2)
        cosine_similarity = dot_prod / (mag1 * mag2)
        cosine_similarity = max(-1.0, min(1.0, cosine_similarity))

        return 1.0 - cosine_similarity

    def __len__(self) -> int:
        return len(self.vectors)

    def __repr__(self) -> str:
        has_embed_fn = "Yes" if self._embedding_fn else "No"
        return f"VectorIndex(count={len(self)}, dim={self._vector_dim}, metric='{self._distance_metric}', has_embedding_fn='{has_embed_fn}')"

In [626]:
# BM25 实现
from collections import Counter
from typing import Callable, Any, List, Dict, Tuple


class BM25Index:
    def __init__(
        self,
        k1: float = 1.5,
        b: float = 0.75,
        tokenizer: Optional[Callable[[str], List[str]]] = None,
    ):
        self.documents: List[Dict[str, Any]] = []
        self._corpus_tokens: List[List[str]] = []
        self._doc_len: List[int] = []
        self._doc_freqs: Dict[str, int] = {}
        self._avg_doc_len: float = 0.0
        self._idf: Dict[str, float] = {}
        self._index_built: bool = False

        self.k1 = k1
        self.b = b
        self._tokenizer = tokenizer if tokenizer else self._default_tokenizer

    def _default_tokenizer(self, text: str) -> List[str]:
        text = text.lower()
        tokens = re.split(r"\W+", text)
        return [token for token in tokens if token]

    def _update_stats_add(self, doc_tokens: List[str]):
        self._doc_len.append(len(doc_tokens))

        seen_in_doc = set()
        for token in doc_tokens:
            if token not in seen_in_doc:
                self._doc_freqs[token] = self._doc_freqs.get(token, 0) + 1
                seen_in_doc.add(token)

        self._index_built = False

    def _calculate_idf(self):
        N = len(self.documents)
        self._idf = {}
        for term, freq in self._doc_freqs.items():
            idf_score = math.log(((N - freq + 0.5) / (freq + 0.5)) + 1)
            self._idf[term] = idf_score

    def _build_index(self):
        if not self.documents:
            self._avg_doc_len = 0.0
            self._idf = {}
            self._index_built = True
            return

        self._avg_doc_len = sum(self._doc_len) / len(self.documents)
        self._calculate_idf()
        self._index_built = True

    def add_document(self, document: Dict[str, Any]):
        if not isinstance(document, dict):
            raise TypeError("Document must be a dictionary.")
        if "content" not in document:
            raise ValueError("Document dictionary must contain a 'content' key.")

        content = document.get("content", "")
        if not isinstance(content, str):
            raise TypeError("Document 'content' must be a string.")

        doc_tokens = self._tokenizer(content)

        self.documents.append(document)
        self._corpus_tokens.append(doc_tokens)
        self._update_stats_add(doc_tokens)

    def add_documents(self, documents: List[Dict[str, Any]]):
        if not isinstance(documents, list):
            raise TypeError("Documents must be a list of dictionaries.")

        if not documents:
            return

        for i, doc in enumerate(documents):
            if not isinstance(doc, dict):
                raise TypeError(f"Document at index {i} must be a dictionary.")
            if "content" not in doc:
                raise ValueError(f"Document at index {i} must contain a 'content' key.")
            if not isinstance(doc["content"], str):
                raise TypeError(f"Document 'content' at index {i} must be a string.")

            content = doc["content"]
            doc_tokens = self._tokenizer(content)

            self.documents.append(doc)
            self._corpus_tokens.append(doc_tokens)
            self._update_stats_add(doc_tokens)

        self._index_built = False

    def _compute_bm25_score(self, query_tokens: List[str], doc_index: int) -> float:
        score = 0.0
        doc_term_counts = Counter(self._corpus_tokens[doc_index])
        doc_length = self._doc_len[doc_index]

        for token in query_tokens:
            if token not in self._idf:
                continue

            idf = self._idf[token]
            term_freq = doc_term_counts.get(token, 0)

            numerator = idf * term_freq * (self.k1 + 1)
            denominator = term_freq + self.k1 * (
                1 - self.b + self.b * (doc_length / self._avg_doc_len)
            )
            score += numerator / (denominator + 1e-9)

        return score

    def search(
        self,
        query: Any,
        k: int = 1,
        score_normalization_factor: float = 0.1,
    ) -> List[Tuple[Dict[str, Any], float]]:
        if not self.documents:
            return []

        if isinstance(query, str):
            query_text = query
        else:
            raise TypeError("Query must be a string for BM25Index.")

        if k <= 0:
            raise ValueError("k must be a positive integer.")

        if not self._index_built:
            self._build_index()

        if self._avg_doc_len == 0:
            return []

        query_tokens = self._tokenizer(query_text)
        if not query_tokens:
            return []

        raw_scores = []
        for i in range(len(self.documents)):
            raw_score = self._compute_bm25_score(query_tokens, i)
            if raw_score > 1e-9:
                raw_scores.append((raw_score, self.documents[i]))

        raw_scores.sort(key=lambda item: item[0], reverse=True)

        normalized_results = []
        for raw_score, doc in raw_scores[:k]:
            normalized_score = math.exp(-score_normalization_factor * raw_score)
            normalized_results.append((doc, normalized_score))

        normalized_results.sort(key=lambda item: item[1])

        return normalized_results

    def __len__(self) -> int:
        return len(self.documents)

    def __repr__(self) -> str:
        return f"BM25VectorStore(count={len(self)}, k1={self.k1}, b={self.b}, index_built={self._index_built})"

In [627]:
# 检索器实现

from typing import Any, List, Dict, Tuple, Protocol, Callable, Optional
import random
import string


class SearchIndex(Protocol):
    def add_document(self, document: Dict[str, Any]) -> None: ...

    # 添加 'add_documents' 方法以避免 VoyageAI 的速率限制错误
    def add_documents(self, documents: List[Dict[str, Any]]) -> None: ...

    def search(self, query: Any, k: int = 1) -> List[Tuple[Dict[str, Any], float]]: ...


class Retriever:
    def __init__(
        self,
        *indexes: SearchIndex,
        reranker_fn: Optional[
            Callable[[List[Dict[str, Any]], str, int], List[str]]
        ] = None,
    ):
        if len(indexes) == 0:
            raise ValueError("At least one index must be provided")
        self._indexes = list(indexes)
        self._reranker_fn = reranker_fn

    def add_document(self, document: Dict[str, Any]):
        if "id" not in document:
            document["id"] = "".join(
                random.choices(string.ascii_letters + string.digits, k=4)
            )

        for index in self._indexes:
            index.add_document(document)

    # 添加 'add_documents' 方法以避免 VoyageAI 的速率限制错误
    def add_documents(self, documents: List[Dict[str, Any]]):
        for index in self._indexes:
            index.add_documents(documents)

    def search(
        self, query_text: str, k: int = 1, k_rrf: int = 60
    ) -> List[Tuple[Dict[str, Any], float]]:
        if not isinstance(query_text, str):
            raise TypeError("Query text must be a string.")
        if k <= 0:
            raise ValueError("k must be a positive integer.")
        if k_rrf < 0:
            raise ValueError("k_rrf must be non-negative.")

        all_results = [index.search(query_text, k=k * 5) for index in self._indexes]

        doc_ranks = {}
        for idx, results in enumerate(all_results):
            for rank, (doc, _) in enumerate(results):
                doc_id = id(doc)
                if doc_id not in doc_ranks:
                    doc_ranks[doc_id] = {
                        "doc_obj": doc,
                        "ranks": [float("inf")] * len(self._indexes),
                    }
                doc_ranks[doc_id]["ranks"][idx] = rank + 1

        def calc_rrf_score(ranks: List[float]) -> float:
            return sum(1.0 / (k_rrf + r) for r in ranks if r != float("inf"))

        scored_docs: List[Tuple[Dict[str, Any], float]] = [
            (ranks["doc_obj"], calc_rrf_score(ranks["ranks"]))
            for ranks in doc_ranks.values()
        ]

        filtered_docs = [(doc, score) for doc, score in scored_docs if score > 0]
        filtered_docs.sort(key=lambda x: x[1], reverse=True)

        result = filtered_docs[:k]

        if self._reranker_fn is not None:
            docs_only = [doc for doc, _ in result]

            for doc in docs_only:
                if "id" not in doc:
                    doc["id"] = "".join(
                        random.choices(string.ascii_letters + string.digits, k=4)
                    )

            doc_lookup = {doc["id"]: doc for doc in docs_only}
            reranked_ids = self._reranker_fn(docs_only, query_text, k)

            new_result = []
            original_scores = {id(doc): score for doc, score in result}

            for doc_id in reranked_ids:
                if doc_id in doc_lookup:
                    doc = doc_lookup[doc_id]
                    score = original_scores.get(id(doc), 0.0)
                    new_result.append((doc, score))

            result = new_result

        return result

In [628]:
# 重排序函数
def reranker_fn(docs, query_text, k):
    joined_docs = "\n".join(
        [
            f"""
        <document>
        <document_id>{doc["id"]}</document_id>
        <document_content>{doc["content"]}</document_content>
        </document>
        """
            for doc in docs
        ]
    )

    prompt = f"""
    你将收到一组文档及各自 id。请从中选出与用户问题最相关的 {k} 篇，并按相关度从高到低排序。

    用户问题：
    <question>
    {query_text}
    </question>
    
    候选文档：
    <documents>
    {joined_docs}
    </documents>

    请严格按以下 JSON 格式回复（仅返回 JSON，不要其他说明）：
    ```json
    {{
        "document_ids": ["id1", "id2", ...]   // 共 {k} 个文档 id，按与问题的相关度从高到低排列，最相关的排在最前
    }}
    ```
    """

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")

    result = chat(messages, stop_sequences=["```"])

    # 注意：已更新为使用 'text_from_message' 辅助函数
    return json.loads(text_from_message(result))["document_ids"]

In [629]:
# 为单个块添加上下文
def add_context(text_chunk, source_text):
    prompt = f"""
    请编写一段简短精炼的文本，用于将这个块定位在整个源文档中，以改善该块的搜索检索效果。

    原始源文档：
    <document> 
    {source_text}
    </document> 

    需要定位的块：
    <chunk> 
    {text_chunk}
    </chunk>
    
    仅回答精炼的上下文内容，不要其他内容。
    """

    messages = []

    add_user_message(messages, prompt)
    result = chat(messages)

    # 注意：已更新为使用 'text_from_message' 辅助函数
    return text_from_message(result) + "\n" + text_chunk

In [630]:
# 读取源文档
with open("./RAG/report_cn.md", "r") as f:
    report_text = f.read()

chunks = chunk_by_section(report_text)

In [631]:
add_context(chunks[5], report_text)

'本段位于《年度跨学科研究综述》的“第二节：软件工程”，紧接第一节医学研究，位于第三节财务分析之前。它详细记录了 Project Phoenix 系统稳定性的技术修复与性能提升，涉及特定错误代码（如 `ERR_MEM_ALLOC_FAIL_0x8007000E`）及事件 `INC-2023-Q4-011`，其结论直接关联第六节产品工程的依赖关系及第十节的网络安全分析。\n第二节：软件工程——Project Phoenix 稳定性提升\n\n软件工程部门投入大量精力提升支撑 Project Phoenix 的核心系统的稳定性与性能。反复出现的问题，尤其是高峰负载下的 `ERR_MEM_ALLOC_FAIL_0x8007000E` 以及影响数据检索的 `TIMEOUT_QUERY_DB_0xDEADBEEF`，被列为优先处理项，对应事件成本为 INC-2023-Q4-011。根因分析指向主数据缓存算法的低效以及数据库索引策略欠佳。补丁部署解决了内存分配错误，在 2024 年第四季度模拟压力测试中（测试用例 ID：INC-2023-Q4-011）测得关键故障减少约 40%。查询模块的进一步重构已安排在下一发布周期，旨在解决超时问题。这些发现凸显了健全测试流程的重要性，尤其考虑到产品工程团队（第六节）所识别的依赖关系。团队继续密切监控系统遥测，以发现任何回归或新出现的错误模式。2024 年第四季度，团队还协助处理了 INC-2023-Q4-011 事件。\n'

In [632]:
# 创建向量索引、BM25 索引，然后使用它们创建检索器
vector_index = VectorIndex(embedding_fn=generate_embedding)
bm25_index = BM25Index()

retriever = Retriever(bm25_index, vector_index, reranker_fn=reranker_fn)

In [619]:
# 为每个块添加上下文，然后添加到检索器
num_start_chunks = 2
num_prev_chunks = 2
contextualized_chunks = []

for i, chunk in enumerate(chunks):
    context_parts = []

    # 文档开头的初始块集合
    context_parts.extend(chunks[: min(num_start_chunks, len(chunks))])

    # 当前正在添加上下文的块之前的额外块
    start_idx = max(0, i - num_prev_chunks)
    context_parts.extend(chunks[start_idx:i])

    context = "\n".join(context_parts)

    contextualized_chunk = add_context(chunk, context)
    print(contextualized_chunk)

    contextualized_chunks.append(contextualized_chunk)

# 注意：已转换为批量操作以避免 VoyageAI 的速率限制错误
retriever.add_documents([{"content": chunk} for chunk in contextualized_chunks])

此块为报告的总标题及执行摘要开头，概括了涵盖医学、软件工程、财务、法律等十个领域的年度跨学科研究综述与跨部门洞察。
# **年度跨学科研究综述：跨领域洞察**

该文段为《年度跨学科研究综述：跨领域洞察》报告的“执行摘要”，简要概述了过去财年在医学（XDR-471）、软件工程、财务、法律（Synergy Dynamics）、产品工程（Model Zircon-5）、药物研发（CTX-204b）及网络安全等十个关键领域的跨学科研究核心进展与战略洞察。
执行摘要

本报告综合了过去财年组织内各运营与研发部门的主要发现与持续研究进展。我们的优势在于思想与方法的交叉融合，推动创新并应对超越传统学科边界的复杂挑战。本年度的综述突出了十个关键领域的显著进展。**医学研究**方面聚焦罕见病 XDR-471 综合征，取得了新的诊断洞察。与此同时，**软件工程**通过错误代码分析（如 `ERR_MEM_ALLOC_FAIL_0x8007000E`）解决了长期存在的稳定性问题并实施了关键修复。**财务分析**显示季度业绩喜忧参半，促使开展战略审视，尤其关注影响研发管线资源配置的决策。

**科学实验**领域也出现重要进展，对新材料特性进行了表征，可能影响未来产品线。**法律动态**团队应对了复杂判例，特别是在与 _Synergy Dynamics_ 案相关的知识产权方面，确保合规并降低风险。**产品工程**在综合多方反馈后，完成了下一代 Model Zircon-5 的规格定稿。**历史研究**对加尔维斯顿协议（Galveston Accords）的洞察为当前市场动态提供了意外背景。**项目管理**在资源受限情况下成功推进了 Project Cerberus 的关键阶段，并通过详细进度报告记录。**药物研发**基于有前景的生物标志物结果，将化合物 CTX-204b 推进至进一步试验。最后，**网络安全分析**针对复杂威胁采取了应对措施，并基于详细事件取证强化了防御。这些工作的合力凸显了我们一体化方法的价值。

此块是《年度跨学科研究综述：跨领域洞察》的目录，概述了报告的完整结构，提供了从医学研究、软件工程到网络安全分析等十个跨学科章节的导航索引。
目录

1. 执行摘要
2. 目录
3. 方法论
4. 第一节：医学研究——理解 XDR-471 综合征
5. 第二节：软件工程——Proj

In [620]:
results = retriever.search("工程团队是如何处理 INC-2023-Q4-011 的？", 2)

for doc, score in results:
    print(score, "\n", doc["content"][0:200], "\n---\n")

0.03278688524590164 
 本段位于《年度跨学科研究综述：跨领域洞察》的“第二节：软件工程”，衔接第一节医学研究，并与第六节产品工程相关联。它详述了 Project Phoenix 的稳定性提升工作，针对执行摘要中提到的内存分配错误（ERR_MEM_ALLOC_FAIL）及数据库超时问题提供了技术根因分析与修复进度。
第二节：软件工程——Project Phoenix 稳定性提升

软件工程部门投入大量精力提升支撑 Proj 
---

0.03225806451612903 
 此片段是《年度跨学科研究综述：跨领域洞察》的第十节（末尾章节），继第九节（药物研发）之后，详细记录了网络安全运营中心对针对财务部（第三节）及敏感研究数据（第一节、第九节）的定向入侵事件（INC-2023-Q4-011）的防御、遏制及取证分析。
第十节：网络安全分析——事件响应报告：INC-2023-Q4-011

网络安全运营中心成功遏制并修复了编号为 `INC-2023-Q4-011` 的定向入 
---

